## NW London Health Inequalities Project – Technical Walkthrough

### Data Engineering

#### Data Ingestion

The project integrates 30+ public datasets from:

* Office for Health Improvement and Disparities (OHID) Fingertips API
* Office for National Statistics (ONS)
* Greater London Authority (GLA)
* GP Patient Survey (GPPS)
* Ethnicity Facts and Figures

Key Python libraries:

In [1]:
import pandas as pd
import numpy as np
import requests
from pathlib import Path

#### API Ingestion Example

Data was ingested directly from the OHID Fingertips API.

In [ ]:
response = requests.get(api_url)
response.raise_for_status()
data = response.json()

In [ ]:
def fetch_fingertips_indicator(
    indicator_id: int,
    timeout: int = 20,      # fail fast
    retries: int = 1        # keep low on unstable days
) -> pd.DataFrame:
    url = (
        "https://fingertips.phe.org.uk/api/all_data/csv/by_indicator_id"
        f"?indicator_ids={indicator_id}"
    )

    last_exc = None
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, timeout=timeout)
            r.raise_for_status()
            return pd.read_csv(pd.io.common.BytesIO(r.content), low_memory=False)
        except Exception as exc:
            last_exc = exc
            if attempt < retries:
                time.sleep(1)
                continue
            raise last_exc

Why `requests`?

* Automated API retrieval
* Repeatable data acquisition
* Reduced manual downloads
* Supports scalable pipelines


requests → API call → JSON/data response → Pandas DataFrame → Parquet file

Why Parquet?

Processed datasets were stored as Parquet files.

Benefits:

* Smaller file sizes
* Faster loading than CSV
* Preserves data types
* Creates a stable analytical layer independent of source APIs

In [ ]:
PARQUET_ENGINE = "pyarrow" if has_module("pyarrow") else "fastparquet"
print("Parquet engine:", PARQUET_ENGINE)

In [ ]:
OUT_PARQUET = "_processed_fingertips_nwl_selected_indicators.parquet"

# Convert mixed-type label columns to a single consistent type for Parquet
for col in ["Age", "Time period", "Sex", "Category", "Recent Trend", "Compared to England"]:
    if col in data.columns:
        data[col] = data[col].astype("string")

data.to_parquet(HERE / OUT_PARQUET, index=False, engine=PARQUET_ENGINE)

print("Saved:", OUT_PARQUET)



Typical workflow:

API/CSV/Excel → Pandas → Parquet → Audit → Harmonisation → Modelling

### Data Processing & Harmonisation

The harmonisation notebook aligned multiple datasets into a consistent analytical structure.

#### Borough Standardisation

e.g. "Hammersmith and Fulham" vs. "Hammersmith & Fulham"

Purpose:

* Consistent borough naming
* Reliable joins across datasets
* Improved data quality

In [ ]:
# Borough name standardisation
BOROUGH_ALIASES = {
    "Hammersmith & Fulham": "Hammersmith and Fulham",
    "Kensington & Chelsea": "Kensington and Chelsea",
}

def normalise_borough(name: str) -> str:
    name = str(name).strip()
    return BOROUGH_ALIASES.get(name, name)

#### Time Standardisation

Different datasets stored time differently.

e.g. model_time:
- 2011 - 13
- 2021
- Year 2019
- 2019/20

Purpose:
* Align all datasets to a common analytical timeline

In [ ]:
# Time Standardisation
def extract_year(value):
    if pd.isna(value):
        return np.nan

    text = str(value)

    match = re.search(
        r"(19|20)\d{2}",
        text
    )

    if match:
        return int(match.group())

    return np.nan

In [ ]:
# Time Standardisation application
df_fingertips_std["model_year"] = (
    df_fingertips_std["model_time"]
    .apply(extract_year)
)

#### Data Completeness Audit
Purpose:

* Identify missing values
* Assess borough-level coverage
* Evaluate dataset readiness

In [ ]:
# Metric availability by borough
availability_df = (
    df_aligned
    .groupby("area_name")
    .agg(
        total_rows=("value", "size"),
        available_values=("value", lambda x: x.notna().sum()),
    )
    .reset_index()
)

availability_df["coverage_rate"] = (
    availability_df["available_values"]
    / availability_df["total_rows"]
)

availability_df = availability_df.sort_values(
    ["coverage_rate", "area_name"],
    ascending=[False, True]
).reset_index(drop=True)

availability_df

Data Coverage Total:

completeness_df
total_rows	rows_with_values	coverage_rate
9157	    5325	            0.581522

Data Coverage by Borough:

area_name	                coverage_rate
0	Ealing	                0.599665
1	Brent	                0.599332
2	Hillingdon	            0.594915
3	Hounslow                0.593883
4	Harrow	                0.588286
5	Westminster	            0.566396
6	Hammersmith and Fulham	0.564663
7	Kensington and Chelsea	0.537428


### Health Inequalities Modelling

The modelling notebook transformed multiple indicators into a single framework.

#### Z-Score Standardisation


Different indicators operate on different scales:

| Indicator               | Scale            |
| ----------------------- | ---------------- |
| Smoking prevalence      | %                |
| Obesity rate            | %                |
| Hospital admissions     | Rate per 100,000 |
| Healthy Life Expectancy | Years            |


Purpose:
To make indicators comparable

* Mean = 0
* Standard deviation = 1
* Enables fair comparison across metrics

In [ ]:
# Metric standardisation (z-score)
df_standardised = df_aligned.copy()

valid_values_mask = df_standardised["value"].notna()

metric_stats = (
    df_standardised.loc[valid_values_mask]
    .groupby("metric")["value"]
    .agg(["mean", "std"])
    .reset_index()
    .rename(columns={"mean": "metric_mean", "std": "metric_std"})
)

df_standardised = df_standardised.merge(
    metric_stats,
    on="metric",
    how="left"
)

df_standardised["z_score"] = np.where(
    df_standardised["metric_std"].notna()
    & (df_standardised["metric_std"] != 0),
    (
        df_standardised["value"]
        - df_standardised["metric_mean"]
    ) / df_standardised["metric_std"],
    np.nan
)

In [ ]:
# Standardisation check
zscore_summary_df = (
    df_standardised["z_score"]
    .describe()
    .to_frame(name="z_score_stats")
)

zscore_summary_df

| Statistic |   z_score |
| --------- | --------: |
| Count     |      5325 |
| Mean      |  0.000000 |
| Std       |  0.996425 |
| Min       | -3.967924 |
| 25%       | -0.734507 |
| 50%       | -0.157806 |
| 75%       |  0.599343 |
| Max       |  6.451899 |


#### Value Direction Alignment

Purpose:

* Align all indicators to the same interpretation
* Higher scores consistently represent worse outcome/higher health burden



Not all indicators move in the same direction.

Examples:

| Indicator               | Direction |
| ----------------------- | --------- |
| Smoking prevalence      | Adverse   |
| Suicide rate            | Adverse   |
| Healthy Life Expectancy | Positive  |
| Screening coverage      | Positive  |

In [ ]:
# Metric direction mapping:
metric_direction_map = {
    "Healthy Life Expectancy": "positive",
    "Life Expectancy": "positive",
    "Screening": "positive",
    "Vaccination": "positive",
    "Smoking": "adverse",
    "Obesity": "adverse",
    "Suicide": "adverse",
}

In [ ]:
# Determine direction for each metric:
def classify_metric_direction(metric_name):
    metric_text = str(metric_name)

    for keyword, direction in metric_direction_map.items():
        if keyword.lower() in metric_text.lower():
            return direction

    return "adverse"

In [ ]:
# Apply direction function to the dataframe
df_direction = df_standardised.copy()

df_direction["direction"] = (
    df_direction["metric"]
    .apply(classify_metric_direction)
)

In [ ]:
# Direction alignment/adjustment:
df_direction["adjusted_z_score"] = np.where(
    df_direction["direction"] == "positive",
    -df_direction["z_score"],
    df_direction["z_score"]
)

### Final Outputs


#### NW London Borough Inequality Typology

The table below summarises borough-level inequality typologies derived from combined health outcomes, prevention indicators, population structure, and service-access patterns across North West London.

#### Interpretation

- Higher inequality scores indicate greater overall inequality burden.
- Preventive inequality score reflects upstream prevention-related risk factors.
- Trend change reflects directional movement over time.
- Recommended actions were generated from borough typology clustering and inequality patterns.

| Borough | Borough Typology | Recommended Action | Inequality Score | Preventive Inequality Score | Black Population Share | Trend Change |
|---|---|---|---:|---:|---:|---:|
| Harrow | Emerging risk | Early intervention and close monitoring | -0.196 | 0.282 | 0.073 | 0.806 |
| Hammersmith and Fulham | High priority: equity + prevention | Targeted outreach, VCSE partnership, prevention | 0.131 | 0.150 | 0.123 | -0.955 |
| Brent | High priority: equity + prevention | Targeted outreach, VCSE partnership, prevention | 0.114 | 0.267 | 0.175 | -1.958 |
| Ealing | High priority: equity + prevention | Targeted outreach, VCSE partnership, prevention | 0.034 | 0.064 | 0.108 | -1.005 |
| Westminster | High priority: outcomes/access | Improve access, pathways, and service responsiveness | 0.009 | -0.454 | 0.081 | -0.684 |
| Hounslow | High priority: prevention | Strengthen early intervention and prevention | 0.147 | 0.154 | 0.072 | -0.691 |
| Hillingdon | Lower burden / monitor | Maintain performance and monitor | -0.033 | -0.107 | 0.078 | -0.414 |
| Kensington and Chelsea | Lower burden / monitor | Maintain performance and monitor | -0.247 | -0.357 | 0.079 | 0.055 |


### Final Notes

#### About this project 

Iterative project narure - 3 phases so far.
- Phase 1: identify determinents and drivers across NW London
- Phase 2: borough level health ineaquality differentiators
- Phase 3: more granular geography - ward and LSOA level - to provide a more targeted intervention planning.

This project demonstrates an end-to-end data workflow rather than just health analysis. I identified and ingested data from more than 30 sources, standardised and harmonised them into a common analytical structure, and developed a modelling framework to support evidence-based decision making.

#### Transferrable skills

Skills like - integrating fragmented data and turning it into actionable insight - is common across healthcare, finance, retail and the public sector.

Retail:
- Combine customer, transaction and loyalty data
- Identify drivers of churn
- Improve customer retention

Financial Services:
- Combine customer, product and risk data
- Detect high-risk segments
- Improve risk management

Public Sector:
- Combine demographic, service usage and deprivation data
- Prioritise communities needing support

#### Project imnprovements:

- From notebook logic to modular Python scripts and automated pipeline

- Using Databricks, programme Notebook run nightly so outputs arte updated continuously

- Predictive modelling to identify emerging inequality risks before health outcomes deteriorate.
